# 2.4 Ascend C 的 Hello World
本入门示例基于 Ascend C SIMD 实现 Hello World 算子，带你快速上手实践，涵盖 Device 端核函数实现、Host 端调用以及编译运行的完整流程，帮助开发者建立整体认知。

**功能介绍**：在 NPU 上打印 `Hello World!!!`。


## Device 端 Kernel 实现
后缀名为 `*.asc` 的代码文件同时包含 Host 端与 Device 端代码，其 Device 端核函数实现如下：

```cpp
__global__ __vector__ void hello_world()
{
    AscendC::printf("Hello World!!!\n");
}
```

> SIMD 算子的 Kernel 函数需要额外的修饰符，`__global__` 表明可被 Host 调用，`__vector__` 说明该算子在矢量计算单元上执行。


## Host 端代码实现
Host 端通过 `<<<>>>` 语法糖调用 Device 端核函数：

```cpp
hello_world<<<8, nullptr, stream>>>();
```

其中 `8` 为启动的核数（blockDim），各核并行执行同一份核函数代码。


In [ ]:
!mkdir -p Sources/02.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment ready.")


In [ ]:
%%writefile Sources/02.04/hello_world.asc

#include "basic_api/kernel_operator_dump_tensor_intf.h"
#include "acl/acl.h"

__global__ __vector__ void hello_world()
{
    AscendC::printf("Hello World!!!\n");
}

int main(int argc, char const* argv[])
{
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    hello_world<<<8, nullptr, stream>>>();
    aclrtSynchronizeStream(stream);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}


## 编译与运行
用毕昇编译器编译，本课程面向 950 使用 `--npu-arch=dav-3510`：


In [ ]:
!bisheng Sources/02.04/hello_world.asc --npu-arch=dav-3510 -o Sources/02.04/demo


In [ ]:
!if [ -e /dev/davinci0 ]; then Sources/02.04/demo; else cannsim record Sources/02.04/demo -s Ascend950 -o Sources/02.04/sim_out; fi


执行本样例，将打印 8 行 `Hello World!!!`，对应 8 个 AIV Core 并行执行。


## 课后实践
修改核函数，使其打印当前核编号（提示：`asc_get_block_idx()`），并在 4 个核上运行。


In [ ]:
!cat ./answer/02.04_answer.txt
